# 年报处理 重构

## 元数据
- 原始路径: 代码库/模版/年报处理
- 创建时间: 2026-02-15
- 任务ID: T38

## 1. 项目概述

### 1.1 功能描述
年报处理项目专注于收集、解析和分析企业年度报告，包括财务报表、管理层讨论与分析、公司治理等信息。

核心功能：
- 从企业预警通网站爬取年报PDF
- 批量OCR处理PDF文件
- 使用大模型(通义千问)提取关键财务信息
- 数据库存储与管理

### 1.2 数据源
- 企业预警通API (https://www.qyyjt.cn)
- 巨潮资讯 (http://www.cninfo.com.cn)
- MySQL数据库 (yq库)

### 1.3 输出结果
- 年报PDF文件
- OCR识别文本
- 财务信息提取结果(股东分红、资本公积增加等)

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import re
import json
import time
import random
import shutil
import urllib.parse
from datetime import datetime, date, timedelta
from pathlib import Path
from time import sleep

# 第三方库
import pandas as pd
import numpy as np
import requests
import sqlalchemy
from sqlalchemy import text
import openai
import fitz  # PyMuPDF

# 导入配置
from config import *

print("依赖导入完成")

### 2.2 配置参数

In [ ]:
# 创建数据库引擎
def create_sql_engine():
    """创建MySQL数据库连接引擎"""
    connection_string = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
    engine = sqlalchemy.create_engine(
        connection_string,
        poolclass=sqlalchemy.pool.NullPool
    )
    return engine

# 创建大模型客户端
def create_llm_client():
    """创建通义千问API客户端"""
    client = openai.OpenAI(
        api_key=DASHSCOPE_API_KEY,
        base_url=DASHSCOPE_BASE_URL
    )
    return client

# 初始化
sql_engine = create_sql_engine()
llm_client = create_llm_client()

print("数据库连接和LLM客户端初始化完成")

## 3. 数据获取

### 3.1 数据源连接

In [ ]:
def query_company_list(sql_engine):
    """从数据库查询公司列表"""
    sql = """
    SELECT A.trade_code, A.fileName, A.公司名称 as company
    FROM yq.23年财报文件 A
    WHERE A.fileName != ''
    """
    with sql_engine.begin() as connection:
        df = pd.read_sql(sql, connection)
    return df

# 查询公司列表
df_companies = query_company_list(sql_engine)
print(f"共查询到 {len(df_companies)} 家公司")

### 3.2 企业预警通API

In [ ]:
def search_company_code(company_name):
    """通过企业预警通API搜索公司代码"""
    encoded_name = urllib.parse.quote(company_name)
    url = f"{QYYJT_API_URL}?root_type=securities&skip=0&pagesize=15&text={encoded_name}"
    
    headers = {
        'authority': 'www.qyyjt.cn',
        'method': 'POST',
        'scheme': 'https',
        'accept': '*/*',
        'client': 'pc-web;pro',
        'content-type': 'application/x-www-form-urlencoded',
        'origin': 'https://www.qyyjt.cn',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    data = {
        "root_type": "securities",
        "skip": 0,
        "pagesize": 15,
        "text": company_name
    }
    
    try:
        r = requests.post(url, headers=headers, data=urllib.parse.urlencode(data))
        result = json.loads(str(r.content, encoding="utf-8"))
        if result.get('success') and result.get('data', {}).get('list'):
            return result['data']['list'][0]['code']
    except Exception as e:
        print(f"搜索公司代码失败: {e}")
    return None

print("企业预警通API函数定义完成")

In [ ]:
def get_annual_report_list(company_code, company_name):
    """获取公司年报列表"""
    url = f"{QYYJT_API_URL}?exportFlag=false&aggs=1&area=&associITCode={company_code}&classify=8147,8219,8146,8218,9220,9155,9221,9156,8144,9218,8216,9219,9159,8148,8220,8354,8325,8353,8324,8177&title=&content=&optionalCombination=&contentMark=content&from=0&industry=&isImportNotice=false&market=&publishDate=&query=&scope=6&size=50&sort=1&trcode="
    
    headers = {
        'authority': 'www.qyyjt.cn',
        'method': 'GET',
        'scheme': 'https',
        'accept': '*/*',
        'client': 'pc-web;pro',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        r = requests.post(url, headers=headers)
        result = json.loads(str(r.content, encoding="utf-8"))
        
        # 过滤年报
        pattern1 = r"专项审计|摘要|page|风险提示|提示性|补充公告|更正|鉴证报告|代理事务|增信|担保|监管|说明"
        pattern2 = r"2023"
        pattern3 = r"审计|财务|年度|年报"
        
        for item in result.get('data', {}).get('hits', []):
            if item.get('tag', {}).get('label') == '年报':
                file_info = item.get('file', [{}])[0]
                file_name = file_info.get('fileName', '')
                if (re.search(pattern3, file_name) and 
                    re.search(pattern2, file_name) and 
                    not re.search(pattern1, file_name)):
                    return file_info.get('fileUrl'), file_name
    except Exception as e:
        print(f"获取年报列表失败: {e}")
    return None, None

print("年报列表获取函数定义完成")

### 3.3 下载年报PDF

In [ ]:
def sanitize_filename(filename):
    """清理文件名，使其符合Windows文件命名规范"""
    sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
    return sanitized

def download_pdf(file_url, file_name, folder_path):
    """下载PDF文件"""
    file_path = os.path.join(folder_path, file_name + ".pdf")
    
    # 检查文件是否已存在
    if os.path.exists(file_path):
        print(f"{file_name} 文件已存在")
        return file_path
    
    try:
        r = requests.get(file_url)
        with open(file_path, "wb") as f:
            f.write(r.content)
        print(f"下载完成: {file_name}.pdf")
        return file_path
    except Exception as e:
        print(f"下载失败: {e}")
        return None

print("PDF下载函数定义完成")

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def check_pdf_content(pdf_path):
    """检查PDF文件是否包含需要的财务内容"""
    doc = fitz.open(pdf_path)
    has_capital_reserve = False
    has_retained_earnings = False
    
    for page_num in range(doc.page_count):
        page = doc[page_num]
        page_text = page.get_text('blocks')
        page_text_list = []
        
        for block in page_text:
            text = block[4].replace('\n', '')
            page_text_list.append(text)
        
        combined_text = '行开头19910513'.join(page_text_list)
        
        if '资本公积' in combined_text and '附注' in combined_text:
            has_capital_reserve = True
        if '未分配利润' in combined_text and '附注' in combined_text:
            has_retained_earnings = True
        
        if has_capital_reserve and has_retained_earnings:
            doc.close()
            return True
    
    doc.close()
    return False

print("PDF内容检查函数定义完成")

### 4.2 OCR处理

In [ ]:
def extract_pdf_text(pdf_path):
    """提取PDF文本内容"""
    doc = fitz.open(pdf_path)
    all_text = []
    
    for page_num in range(doc.page_count):
        page = doc[page_num]
        page_text = page.get_text('text')
        all_text.append(page_text)
    
    doc.close()
    return '\n'.join(all_text)

def save_text_to_file(text, output_path):
    """保存文本到文件"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(text)

print("OCR处理函数定义完成")

## 5. 核心逻辑

### 5.1 大模型提取财务信息

In [ ]:
FINANCIAL_EXTRACTION_PROMPT = '''
作为高级财务信息筛选师，您的任务是依据严格的财务准则，精准提取并总结财务报表摘要中不同年份的两个核心财务板块信息：股东分红与资本公积增加。

### 股东分红:
* "项目"列统一标记为"股东分红"，按年份求和汇总。
* 纳入统计：分配、应付、提取、派发的股利、分配红利、分配现金股利等。
* 不考虑"股票股利"。

### 资本公积增加:
* 范围：仅考虑资本公积增加、增长，不考虑资本公积减少。
* "项目"列统一为：资本公积增加。
* "增加原因"列记录增加的主要原因，如"政府现金拨款"、"股权无偿划转"等。

### 时间标识:
* "2023年"（当期/当年/本年）指2023年度数据。
* "2022年"（上一期/上年/去年）指2022年度数据。

### 金额处理:
* 原始金额保留数字，原始金额单位展示其原始单位。
* 金额列转换为"亿"单位。

### 输出格式:
``` 年份|项目|增加原因|金额|原始金额|原始金额单位 ```
'''

def upload_file_to_qwen(client, file_path):
    """上传文件到通义千问"""
    file_object = client.files.create(
        file=Path(file_path),
        purpose="file-extract"
    )
    return file_object.id

def wait_for_file_processing(client, file_id):
    """等待文件处理完成"""
    while True:
        file_list = client.files.list()
        for item in file_list.data:
            if item.id == file_id:
                if item.status == 'processed':
                    return True
        time.sleep(10)

def extract_financial_info(client, file_id):
    """使用大模型提取财务信息"""
    # 等待文件处理完成
    wait_for_file_processing(client, file_id)
    
    try:
        completion = client.chat.completions.create(
            model="qwen-long",
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'system', 'content': f'fileid://{file_id}'},
                {'role': 'user', 'content': FINANCIAL_EXTRACTION_PROMPT}
            ],
            stream=False
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"提取财务信息失败: {e}")
        return ''

print("大模型提取函数定义完成")

### 5.2 解析提取结果

In [ ]:
def parse_extraction_result(summary):
    """解析大模型提取的结果"""
    # 提取表格内容
    pattern = re.compile(r'```(.*)```', re.MULTILINE | re.DOTALL)
    match = pattern.search(summary)
    if match:
        summary = match.group(1)
    
    # 清理内容
    summary = summary.replace('-', '').replace(' ', '').replace('空值', '')
    
    try:
        lines = summary.split('\n')
        lines = [line for line in lines if line.strip()]
        
        if len(lines) < 2:
            return pd.DataFrame()
        
        header = lines[0].split('|')
        data_lines = [line.split('|') for line in lines[1:]]
        
        df = pd.DataFrame(data_lines, columns=header)
        df = df.apply(lambda x: x.str.strip())
        df = df[df['年份'] != '']
        
        return df
    except Exception as e:
        print(f"解析结果失败: {e}")
        return pd.DataFrame()

print("结果解析函数定义完成")

### 5.3 数据库操作

In [ ]:
def update_database_file_info(sql_engine, trade_code, file_url, file_name):
    """更新数据库中的文件信息"""
    sql = """
    UPDATE 23年财报文件
    SET fileUrl = :fileUrl, fileName = :fileName, ocr = '已重下'
    WHERE trade_code = :trade_code
    """
    params = {
        "trade_code": trade_code,
        "fileUrl": file_url,
        "fileName": file_name
    }
    with sql_engine.begin() as connection:
        connection.execute(text(sql), params)

def insert_extraction_result(sql_engine, df_result, trade_code, file_name, company_name, summary):
    """插入提取结果到数据库"""
    if df_result.empty:
        data = {
            'trade_code': [trade_code],
            'fileName': [file_name],
            '公司名称': [company_name],
            '年份': [2023],
            '项目': ['股东分红'],
            '增加原因': ['现金分红'],
            '金额': [0],
            '原始金额': [0],
            '备注': ['提取失败'],
            'summary': [summary]
        }
        df_result = pd.DataFrame(data)
    else:
        df_result['trade_code'] = trade_code
        df_result['fileName'] = file_name
        df_result['公司名称'] = company_name
        df_result['备注'] = '提取成功'
        df_result['summary'] = summary
    
    with sql_engine.begin() as connection:
        df_result.to_sql('23年财报挖掘', connection, if_exists='append', index=False)

print("数据库操作函数定义完成")

## 6. 执行与测试

### 6.1 主流程函数

In [ ]:
def process_annual_reports(sql_engine, llm_client, folder_path, output_folder):
    """主流程：处理年报"""
    # 确保目录存在
    os.makedirs(folder_path, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)
    
    # 查询公司列表
    df_companies = query_company_list(sql_engine)
    total = len(df_companies)
    
    for index, row in df_companies.iterrows():
        print(f"---{index + 1}/{total}")
        
        trade_code = row['trade_code']
        company_name = row['company']
        file_name = row['fileName']
        
        try:
            # 搜索公司代码
            company_code = search_company_code(company_name)
            if not company_code:
                print(f"未找到公司代码: {company_name}")
                continue
            
            # 获取年报URL
            file_url, report_name = get_annual_report_list(company_code, company_name)
            if not file_url:
                print(f"未找到年报: {company_name}")
                continue
            
            # 下载PDF
            pdf_path = download_pdf(file_url, sanitize_filename(report_name), folder_path)
            if not pdf_path:
                continue
            
            # 提取文本
            text = extract_pdf_text(pdf_path)
            txt_path = pdf_path.replace('.pdf', '.txt')
            save_text_to_file(text, txt_path)
            
            # 上传到大模型并提取信息
            file_id = upload_file_to_qwen(llm_client, txt_path)
            summary = extract_financial_info(llm_client, file_id)
            
            # 解析结果
            df_result = parse_extraction_result(summary)
            
            # 保存到数据库
            insert_extraction_result(
                sql_engine, df_result, trade_code, 
                file_name, company_name, summary
            )
            
            # 移动到已处理目录
            shutil.move(pdf_path, output_folder)
            shutil.move(txt_path, output_folder)
            
            # 随机延迟
            sleep(random.uniform(0.5, 1))
            
        except Exception as e:
            print(f"处理失败 {company_name}: {e}")
            continue

print("主流程函数定义完成")

### 6.2 执行主流程

In [ ]:
# 执行主流程（取消注释以运行）
# folder_path = DATA_DIR / 'annual_reports'
# output_folder = OUTPUT_DIR / 'processed'
# process_annual_reports(sql_engine, llm_client, str(folder_path), str(output_folder))

print("执行主流程代码已准备就绪，取消注释以运行")

### 6.3 结果验证

In [ ]:
def check_ocr_status(sql_engine):
    """检查OCR完成状态"""
    sql = """
    SELECT fileName, ocr
    FROM 23年财报文件
    """
    with sql_engine.begin() as connection:
        df = pd.read_sql(sql, connection)
    
    total = len(df)
    ocr_done = len(df[df['ocr'] != ''])
    ocr_pending = len(df[df['ocr'] == ''])
    
    print(f"总文件数: {total}")
    print(f"已完成OCR: {ocr_done}")
    print(f"待OCR: {ocr_pending}")
    print(f"完成率: {ocr_done / total * 100:.2f}%")
    
    return df

# 检查OCR状态
df_status = check_ocr_status(sql_engine)
df_status.head()

## 7. 工具函数（可复用）

In [ ]:
# 工具函数集合

def extract_chinese_characters(s):
    """提取字符串中的中文字符"""
    if s is None:
        return ''
    return re.sub(r'[^\u4e00-\u9fa5]', '', s)

def convert_amount_to_yi(amount, unit):
    """转换金额为亿元单位"""
    amount = float(amount)
    unit = unit.strip()
    
    if unit == '元':
        return amount / 100000000
    elif unit == '万':
        return amount / 10000
    elif unit == '百万':
        return amount / 100
    elif unit == '千元':
        return amount / 100000
    elif unit == '亿':
        return amount
    else:
        return amount

def extract_links_from_pdf(pdf_path):
    """从PDF中提取超链接"""
    doc = fitz.open(pdf_path)
    links = []
    
    for page in doc:
        annots = page.annots()
        if annots:
            for annot in annots:
                if annot.type[0] == 1:  # 链接类型
                    links.append(annot.uri)
    
    doc.close()
    return links

print("工具函数定义完成")

In [ ]:
# 测试工具函数
test_string = "测试ABC公司123"
print(f"提取中文字符: {extract_chinese_characters(test_string)}")

test_amount = 78387466.98
print(f"金额转换: {convert_amount_to_yi(test_amount, '元')}亿")